# ⚠️ LiFE Pulse — Notebook 3: Consumption Anomaly Detector (M5)
### Spike Alert & Load Leakage Detection (HC26-GEDA01)

Trains an **Isolation Forest** and rolling Z-score model to flag unexpected consumption surges (>30% spike or >2.5 std deviations above rolling 7-day baseline).

---

### Step 1: Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, os, joblib, json
from sklearn.ensemble import IsolationForest

BASE_DIR = '/content/drive/MyDrive/lifepulse'
os.makedirs(f'{BASE_DIR}/artifacts', exist_ok=True)

rng = np.random.default_rng(42)
N = 2500

# Features: [daily_mean, daily_max, peak_hour_share, variance, weekend_ratio]
normal_mean = rng.normal(7.5, 2.0, N).clip(2.0, 25.0)
normal_max = normal_mean * rng.uniform(1.4, 2.5, N)
peak_share = rng.uniform(0.20, 0.45, N)
variance = rng.uniform(0.5, 3.0, N)
weekend_ratio = rng.uniform(1.0, 1.35, N)
X_normal = np.column_stack([normal_mean, normal_max, peak_share, variance, weekend_ratio])

# Injected anomalies
num_anom = int(N * 0.03)
X_anom = np.column_stack([
    rng.uniform(25.0, 60.0, num_anom),
    rng.uniform(50.0, 120.0, num_anom),
    rng.uniform(0.60, 0.90, num_anom),
    rng.uniform(10.0, 30.0, num_anom),
    rng.uniform(2.0, 3.5, num_anom)
])
X = np.vstack([X_normal, X_anom])

print('🚀 Training Isolation Forest model...')
model = IsolationForest(n_estimators=100, contamination=0.03, random_state=42)
model.fit(X)

out_path = f'{BASE_DIR}/artifacts/anomaly_model.joblib'
joblib.dump(model, out_path)
print(f'💾 Model saved to: {out_path}')